In [ ]:


# CONFIGURAÇÕES INICIAIS DAS ANÁLISES
# IMPORTAR BIBLIOTECAS ---
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# CAMINHOS ---
SCRIPT_DIR = Path(__file__).resolve().parent
ANALYTICS_DIR = SCRIPT_DIR.parent

# PARA IMPORTAR FUNÇÃO DE EXPORTAR CSV ---
if str(ANALYTICS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYTICS_DIR))
from utils.export_utils import exportar_csv

# ROOT DO PROJETO ---
PROJECT_ROOT = Path(__file__).resolve().parents[3]

# ROOT DOS OUTPUTS ---
OUTPUT_DIR = (PROJECT_ROOT/ "scripts"/ "Analytics"/ "outputs"/ "gold_02")

# SPARK ---
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# PREPARAÇÃO PARA ANALISAR GOLD 02 - Quais perfis profissionais são mais valorizados pelo mercado?
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# CAMINHO DA GOLD 02 ---
caminho_gold_02 = (PROJECT_ROOT/ "Gold"/ "perguntas_negocio"/ "gold_02_perfis_valorizados")

# BUSCAR CSVs GERADOS PELO SPARK ---
arquivos_gold_02 = [
    str(arquivo)
    for arquivo in caminho_gold_02.glob("part-*.csv")
]

# VALIDAR SE EXISTEM ARQUIVOS ---
if not arquivos_gold_02:
    raise FileNotFoundError(
        f"Nenhum arquivo part-*.csv encontrado em: {caminho_gold_02}"
    )

# CARREGAR GOLD 02 ---
df_perfis = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_02)
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# TRATAMENTO DA BASE
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# REMOVER FAIXA SALARIAL INCONSISTENTE ---
"""
A faixa identificada no fluxo como inconsistente é retirada antes da consolidação e do recálculo dos totais. Assim, ela não influencia o denominador nem os percentis salariais calculados nas etapas seguintes.
"""
df_perfis_tratado = (df_perfis
    .filter((F.col("valor") != "de R$ 101/mês a R$ 2.000/mês") | F.col("valor").isNull())
)

# HARMONIZAR ENGENHARIA E ARQUITETURA DE DADOS ---
"""
As nomenclaturas de Engenharia e Arquitetura de Dados são reunidas em uma única família para evitar que variações de rótulo fragmentem o mesmo perfil profissional em contagens e rankings separados.
"""
df_perfis_tratado = (df_perfis_tratado
    .withColumn("cargo_harmonizado",
        F.when(
            F.col("cargo_atual").isin(
                "Engenheiro de Dados/Arquiteto de Dados/Data Engineer/Data Architect",
                "Engenheiro de Dados/Data Engineer/Data Architect",
                "Arquiteto de Dados/Data Architect"
            ),
            "Engenharia e Arquitetura de Dados")
        .otherwise(F.col("cargo_atual")))
)

# CONSOLIDAR CONTAGENS APÓS HARMONIZAÇÃO ---
df_perfis_consolidado = (df_perfis_tratado
    .groupBy("edicao","cargo_harmonizado","nivel","valor")
    .agg(F.sum("contagem").alias("contagem"))
)

# RECALCULAR TOTAL DE RESPONDENTES ---
"""
Depois dos tratamentos, o total é recalculado por edição, cargo harmonizado e nível. Esse total tratado passa a ser o denominador de pct_na_dimensao e da distribuição acumulada usada nos percentis.
"""
janela_perfil = (Window.partitionBy("edicao","cargo_harmonizado","nivel"))

df_perfis_consolidado = (df_perfis_consolidado
    .withColumn("total_respondentes",F.sum("contagem").over(janela_perfil))
    .withColumn("pct_na_dimensao",F.round((F.col("contagem")/ F.col("total_respondentes")) * 100, 2))
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# ORDENAR AS FAIXAS SALARIAIS
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
Como as faixas salariais são categorias textuais, é criada uma ordem numérica crescente. Essa sequência permite acumular os respondentes do menor para o maior intervalo e localizar corretamente os pontos de P50 e P75.
"""
df_perfis_ordenado = (df_perfis_consolidado
    .withColumn("ordem_faixa_salarial",
        F.when(F.col("valor") == "Menos de R$ 1.000/mês", 1)
        .when(F.col("valor") == "de R$ 1.001/mês a R$ 2.000/mês", 2)
        .when(F.col("valor") == "de R$ 2.001/mês a R$ 3.000/mês", 3)
        .when(F.col("valor") == "de R$ 3.001/mês a R$ 4.000/mês", 4)
        .when(F.col("valor") == "de R$ 4.001/mês a R$ 6.000/mês", 5)
        .when(F.col("valor") == "de R$ 6.001/mês a R$ 8.000/mês", 6)
        .when(F.col("valor") == "de R$ 8.001/mês a R$ 12.000/mês", 7)
        .when(F.col("valor") == "de R$ 12.001/mês a R$ 16.000/mês", 8)
        .when(F.col("valor") == "de R$ 16.001/mês a R$ 20.000/mês", 9)
        .when(F.col("valor") == "de R$ 20.001/mês a R$ 25.000/mês", 10)
        .when(F.col("valor") == "de R$ 25.001/mês a R$ 30.000/mês", 11)
        .when(F.col("valor") == "de R$ 30.001/mês a R$ 40.000/mês", 12)
        .when(F.col("valor") == "Acima de R$ 40.001/mês", 13)
    )
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# SELECIONAR PERFIS ELEGÍVEIS - 2025-2026
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
print("\n" + "=" * 100)
print("3. PERFIS ELEGÍVEIS - 2025-2026")
print("=" * 100)

# TAMANHO DA AMOSTRA POR CARGO E NÍVEL ---
amostra_perfis_atual = (df_perfis_ordenado
    .filter(F.col("edicao") == "2025-2026")
    .select("cargo_harmonizado","nivel","total_respondentes")
    .distinct()
)

# CRITÉRIO DE AMOSTRA
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
Pelo menos 20 respondentes por combinação cargo + nível.
"Outra Opção" não representa um perfil profissional identificável.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

perfis_elegiveis_atual = (amostra_perfis_atual
    .filter((F.col("total_respondentes") >= 20) & (F.col("cargo_harmonizado") != "Outra Opção"))
    .orderBy("nivel",F.desc("total_respondentes"))
)
perfis_elegiveis_atual.show(100,truncate=False)
print("Quantidade de perfis elegíveis:",perfis_elegiveis_atual.count())
"""
Na execução anexada, 29 combinações de cargo e nível atenderam aos critérios definidos para a edição 2025-2026. O recorte evita que o ranking seja construído a partir de combinações com amostras abaixo do limite adotado.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# CALCULAR P50 DOS PERFIS
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
print("\n" + "=" * 100)
print("4. FAIXA SALARIAL MEDIANA - P50")
print("=" * 100)

# BASE ANALÍTICA DOS PERFIS ELEGÍVEIS ---
df_perfis_atual = (df_perfis_ordenado
    .filter((F.col("edicao") == "2025-2026") & (F.col("total_respondentes") >= 20) & (F.col("cargo_harmonizado") != "Outra Opção"))
)

# ACUMULAR AS FAIXAS DO MENOR PARA O MAIOR SALÁRIO ---
"""
P50 e P75 são calculados sobre a distribuição das faixas salariais, e não sobre valores individuais de salário. Por isso, as contagens são acumuladas dentro de cada combinação de cargo e nível seguindo a ordem salarial definida anteriormente.
"""
janela_acumulada = (
    Window
    .partitionBy("cargo_harmonizado","nivel")
    .orderBy("ordem_faixa_salarial")
    .rowsBetween(Window.unboundedPreceding,Window.currentRow)
)

df_perfis_acumulado = (df_perfis_atual
    .withColumn("contagem_acumulada", F.sum("contagem").over(janela_acumulada))
    .withColumn("pct_acumulado",F.round((F.col("contagem_acumulada")/ F.col("total_respondentes")) * 100, ))
)

# IDENTIFICAR PRIMEIRA FAIXA QUE ATINGE 50% ---
"""
O P50 é representado pela primeira faixa em que a distribuição acumulada atinge pelo menos 50% dos respondentes. Essa faixa funciona como referência central da remuneração de cada perfil elegível.
"""
janela_percentil = (
    Window
    .partitionBy("cargo_harmonizado","nivel" )
    .orderBy("ordem_faixa_salarial")
)

faixa_p50_perfis = (df_perfis_acumulado
    .filter(F.col("pct_acumulado") >= 50)
    .withColumn("ordem_p50",F.row_number().over(janela_percentil))
    .filter(F.col("ordem_p50") == 1)
    .select("cargo_harmonizado","nivel","total_respondentes",
        F.col("valor").alias("faixa_salarial_mediana"),
        F.col("ordem_faixa_salarial").alias("ordem_faixa_mediana")
    )
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# CALCULAR P75 DOS PERFIS
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
print("\n" + "=" * 100)
print("5. FAIXA SALARIAL P75")
print("=" * 100)
"""
O P75 identifica a primeira faixa que alcança 75% da distribuição acumulada. Ele complementa a mediana e permite diferenciar perfis que apresentam o mesmo P50, sem transformar as faixas em salários pontuais.
"""

faixa_p75_perfis = (df_perfis_acumulado
    .filter(F.col("pct_acumulado") >= 75)
    .withColumn("ordem_p75",F.row_number().over(janela_percentil))
    .filter(F.col("ordem_p75") == 1)
    .select("cargo_harmonizado","nivel","total_respondentes",
        F.col("valor").alias("faixa_salarial_p75"),
        F.col("ordem_faixa_salarial").alias("ordem_faixa_p75")
    )
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# CONSOLIDAR P50 E P75
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
resumo_salarial_atual = (faixa_p50_perfis
    .join(faixa_p75_perfis,
        on=["cargo_harmonizado","nivel","total_respondentes"],
        how="inner"
    )
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# RANKING SALARIAL DOS PERFIS POR SENIORIDADE
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
print("\n" + "=" * 100)
print("7. RANKING SALARIAL DOS PERFIS POR SENIORIDADE - 2025-2026")
print("=" * 100)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
P50 = CRITÉRIO PRINCIPAL
P75 = CRITÉRIO DE DESEMPATE
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
O ranking é calculado separadamente dentro de cada nível de senioridade, evitando misturar diferenças de cargo com o efeito esperado da progressão de carreira. O dense_rank mantém a mesma posição para perfis com P50 e P75 equivalentes.
"""

janela_ranking = (
    Window
    .partitionBy("nivel")
    .orderBy(F.desc("ordem_faixa_mediana"),F.desc("ordem_faixa_p75"))
)

ranking_perfis_atual = (resumo_salarial_atual
    .withColumn("ranking", F.dense_rank().over(janela_ranking))
    .select("nivel","ranking","cargo_harmonizado","total_respondentes","faixa_salarial_mediana","faixa_salarial_p75","ordem_faixa_mediana","ordem_faixa_p75")
    .orderBy("nivel","ranking",F.desc("total_respondentes"))
)
ranking_perfis_atual.show(100,truncate=False)
"""
Na execução de 2025-2026, Engenheiro de Machine Learning/ML Engineer/AI Engineer ocupa a 1ª posição em Especialista/Staff+ e Sênior. No nível Júnior, Cientista de Dados/Data Scientist aparece em 1º; em Pleno, Cientista de Dados, Engenharia e Arquitetura de Dados e Analytics Engineer compartilham a 1ª posição por apresentarem o mesmo P50 e P75.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# RESULTADO FINAL
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
perfis_valorizados_atual = (
    ranking_perfis_atual
    .select("nivel", "ranking", "cargo_harmonizado", "total_respondentes", "faixa_salarial_mediana", "faixa_salarial_p75", "ordem_faixa_mediana", "ordem_faixa_p75")
    .orderBy("nivel","ranking",F.desc("total_respondentes"))
)

print("\n" + "=" * 100)
print("8. PERFIS MAIS VALORIZADOS - 2025-2026")
print("=" * 100)
perfis_valorizados_atual.show(100, truncate=False)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# EXPORTAÇÃO PARA VISUALIZAÇÃO
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# PERFIS VALORIZADOS.CSV
exportar_csv(perfis_valorizados_atual,OUTPUT_DIR,"perfis_valorizados_atual.csv")
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------